# UCS420: Cognitive Computing
## Assignment 4 — A Cognitive FAQ System Using Pandas (Nova 2.0)

**Name:** Krish Marwaha  
**Roll Number:** 1024170223

This notebook solves Questions 1–6 from the assignment using Pandas.

## Q1. Build Your Personalized Knowledge Base
Create exactly six FAQ entries: four fixed entries and two personalized entries derived from the last two digits of the roll number.

In [ ]:
import pandas as pd

roll_number = "1024170223"
last_two_digits = [int(d) for d in roll_number[-2:]]
categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = []
for digit in last_two_digits:
    category = categories[digit % 3]
    if category == "account":
        personalized_entries.append({
            "question": "how can i update my registered mobile number",
            "answer": "Open account settings, choose Profile, and update your registered mobile number after verification.",
            "keywords": "mobile number profile update",
            "category": category
        })
    elif category == "billing":
        personalized_entries.append({
            "question": "how can i check my latest payment status",
            "answer": "Open the Payments section and select your latest transaction to view its status.",
            "keywords": "payment status transaction billing",
            "category": category
        })
    else:
        personalized_entries.append({
            "question": "where can i find information about available services",
            "answer": "Visit the Help Centre to view information about the available services.",
            "keywords": "services information help centre",
            "category": category
        })

df = pd.DataFrame(fixed_entries + personalized_entries,
                  columns=["question", "answer", "keywords", "category"])
print("Last two digits:", last_two_digits)
print(df.to_string(index=False))


**Personalization check:** The last two digits are `23`. Digit `2` maps to `general` (`2 % 3 = 2`), and digit `3` maps to `billing` (`3 % 3 = 0`). The DataFrame therefore contains exactly six entries.

## Q2. Generate and Score a Hypothesis
The function scores an entry by counting query words that occur in its question, answer, or keywords. Results are returned in descending score order.

In [ ]:
import re

def score_query(query, df):
    query_words = set(re.findall(r"\\b\\w+\\b", query.lower()))
    scored = df.copy()
    scored["score"] = scored.apply(
        lambda row: sum(
            word in (row["question"] + " " + row["answer"] + " " + row["keywords"]).lower()
            for word in query_words
        ),
        axis=1
    )
    return scored[scored["score"] > 0].sort_values(
        by="score", ascending=False, kind="stable"
    )

query = "how can I pay the fee"
matches = score_query(query, df)
print(matches.to_string(index=False) if not matches.empty else "No matching FAQ found.")


## Q3. Find Entries in the Same Category
The function returns all FAQ entries that belong to the supplied category. Here, it is called with the category of the first personalized entry.

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

selected_category = personalized_entries[0]["category"]
print("Selected category:", selected_category)
print(same_category(selected_category, df).to_string(index=False))


## Q4. Add a Keyword and Save the DataFrame
Choose one entry, ask for a new keyword, append it to that entry's keywords, and save all six entries to a CSV file named with the roll number. The fallback keyword makes the notebook runnable in non-interactive environments; enter a keyword when prompted.

In [ ]:
entry_index = 0
new_keyword = input("Enter a new keyword to add to the first FAQ entry: ").strip()

if not new_keyword:
    new_keyword = "membership"

existing_keywords = df.at[entry_index, "keywords"].split()
if new_keyword.lower() not in [word.lower() for word in existing_keywords]:
    df.at[entry_index, "keywords"] = df.at[entry_index, "keywords"] + " " + new_keyword

csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print("Updated entry:")
print(df.iloc[[entry_index]].to_string(index=False))
print(f"Saved updated DataFrame to {csv_filename}")


## Q5. Count FAQ Entries per Category
Use `groupby` to count the entries in each category.

In [ ]:
category_counts = df.groupby("category").size().reset_index(name="count")
print(category_counts.to_string(index=False))


## Q6. Show All Highest-Scoring Ties
This version prints every entry tied for the highest positive score. If no entry matches, it reports that no match was found. The first demonstration uses `fee`, which should tie the two fixed billing questions; the second uses a more specific query.

In [ ]:
def score_query_with_ties(query, df):
    scored = score_query(query, df)
    if scored.empty:
        print("No matching FAQ found.")
        return scored

    highest_score = scored["score"].max()
    best_matches = scored[scored["score"] == highest_score]

    if len(best_matches) > 1:
        print(f"Tie: {len(best_matches)} entries share the highest score ({highest_score}).")
    else:
        print(f"One highest-scoring entry (score {highest_score}).")

    print(best_matches.to_string(index=False))
    return best_matches

print("Demonstration 1 — query: 'fee'")
tie_matches = score_query_with_ties("fee", df)

print("\\nDemonstration 2 — query: 'reset password login'")
single_match = score_query_with_ties("reset password login", df)


## Conclusion
- Built a six-entry Pandas FAQ knowledge base using the last two roll-number digits.
- Implemented query scoring and category filtering.
- Added a keyword and saved the updated DataFrame as a CSV.
- Counted FAQs per category with `groupby`.
- Updated scoring to display every highest-scoring tie.